# Operating Area → Gazetteer Review

Loads all **active** vessels from Supabase, runs each vessel's free-text `operating_area`
through the [Marine Regions](https://marineregions.org) gazetteer, and lets you eyeball the
matches on a Leaflet map and in a side-by-side table.

Deps (all present in this env): `pandas`, `requests`, `folium`, `python-dotenv`.
The lookup loop hits the gazetteer once per unique phrase (cached + throttled) — ~2-3 min the first run.


In [2]:
import os, re, time, requests
import pandas as pd
from pathlib import Path

# locate .env.local by walking up from the notebook's cwd to the repo root
def find_env(name='.env.local'):
    d = Path.cwd()
    for p in [d, *d.parents]:
        if (p / name).exists():
            return p / name
    raise FileNotFoundError(f'{name} not found in {Path.cwd()} or any parent')

env = {}
for line in find_env().read_text().splitlines():
    m = re.match(r'^([A-Z_]+)=(.*)$', line)
    if m:
        env[m.group(1)] = m.group(2).strip().strip('"').strip("'")

SUPABASE_URL = env['NEXT_PUBLIC_SUPABASE_URL']
SERVICE_KEY  = env['SUPABASE_SERVICE_ROLE_KEY']
HEADERS = {'apikey': SERVICE_KEY, 'Authorization': f'Bearer {SERVICE_KEY}'}

# fetch all active vessels, all columns
r = requests.get(f'{SUPABASE_URL}/rest/v1/vessels', headers=HEADERS,
                 params={'select': '*', 'status': 'eq.active', 'limit': '2000'})
r.raise_for_status()
df = pd.DataFrame(r.json())
print('active vessels:', df.shape[0], '| columns:', df.shape[1])
df[['id', 'name', 'country', 'port_city', 'operating_area']].head(20)


active vessels: 565 | columns: 136


,id,name,country,port_city,operating_area
0,752,Teleost (CCGS),Canada,St. John's,Newfoundland
1,236,L'Europe,France,La Seyne-sur-Mer,Mediterranean
2,253,Alkor,Germany,Kiel,None
3,481,Shoyo,Japan,Tokyo,Northwest Pcific Ocean
4,1097,Nautilus,Canada,Kingstown,None
5,1184,Western Flyer,USA,Monterey,None
6,1021,Gloria Michelle,USA,Woods Hole,None
7,1176,Western Shore,Canada,None,Western North America
8,376,Takuyo,Japan,Tokyo,Northwest Pcific Ocean
9,536,Oceania,Poland,Gdansk,"The Baltic, Norwegian, Greenland, Barents Seas"


In [6]:
df[df['operating_area'] != None]['id', 'name', 'country', 'port_city', 'operating_area']



KeyError: ('id', 'name', 'country', 'port_city', 'operating_area')

In [10]:
# --- gazetteer lookup with smart record selection ---

# preferred whole-region place types, best first; "Marine Region" (country-parts) ranked last
PRIORITY = ['IHO Sea Area', 'Large Marine Ecosystem', 'SeaVoX SeaArea',
            'Marine Province', 'Ocean', 'Sea', 'Gulf', 'Bay', 'Strait', 'Channel']
GOOD = set(PRIORITY) | {'Marine Region'}

# drop generic single tokens that match junk
STOP = {'east','west','north','south','open','coast','offshore','nearshore',
        'worldwide','global','region','area','waters','coastal'}

def split_phrases(text):
    if not text:
        return []
    text = re.sub(r'<[^>]+>', ' ', str(text))
    parts = re.split(r'[,;.]|\band\b|/|\(|\)', text, flags=re.I)
    out = []
    for p in parts:
        p = p.strip()
        if 2 < len(p) < 60 and not re.fullmatch(r'[0-9\s]+', p) and p.lower() not in STOP:
            out.append(p)
    return out

def _score(phrase, rec):
    pt   = rec.get('placeType')
    name = (rec.get('preferredGazetteerName') or '').lower()
    exact     = 0 if name == phrase.lower() else 1            # exact name wins
    prio      = PRIORITY.index(pt) if pt in PRIORITY else 50  # then place-type priority
    if pt == 'Marine Region': prio = 40                       # country-parts after whole regions
    has_bbox  = 0 if rec.get('minLatitude') is not None else 1
    return (exact, prio, has_bbox, len(name))

def best_record(phrase, records):
    cands = [x for x in records if x.get('placeType') in GOOD]
    if not cands:
        return None
    cands.sort(key=lambda x: _score(phrase, x))
    return cands[0]

def gazetteer(phrase):
    url = f'https://www.marineregions.org/rest/getGazetteerRecordsByName.json/{requests.utils.quote(phrase)}/true/true/'
    try:
        resp = requests.get(url, headers={'Accept': 'application/json'}, timeout=20)
        if resp.status_code == 404:
            return []
        resp.raise_for_status()
        return resp.json()
    except Exception:
        return []

_cache = {}
def lookup_phrase(phrase):
    key = phrase.lower()
    if key in _cache:
        return _cache[key]
    best = best_record(phrase, gazetteer(phrase))
    res = None
    if best:
        res = {
            'matched_name': best.get('preferredGazetteerName'),
            'place_type':   best.get('placeType'),
            'mrgid':        best.get('MRGID'),
            'bbox': [best.get('minLongitude'), best.get('minLatitude'),
                     best.get('maxLongitude'), best.get('maxLatitude')],
        }
    _cache[key] = res
    time.sleep(0.2)  # throttle the public API
    return res

print('helpers ready')


helpers ready


In [11]:
from pathlib import Path

RESULTS_PATH = Path.cwd() / 'gazetteer_results.pkl'

In [12]:

# --- run gazetteer lookup over every active vessel with an operating_area ---
if RESULTS_PATH.exists():
    results_df = pd.read_pickle(RESULTS_PATH)
    print(f'loaded cached results: {len(results_df)} rows from {RESULTS_PATH}')
    print('(delete that file to force a fresh API run)')
else:
    oa = df[df['operating_area'].notna() & (df['operating_area'].astype(str).str.strip() != '')]
    print('vessels with operating_area:', len(oa))

    rows = []
    for i, (_, v) in enumerate(oa.iterrows()):
        phrases = split_phrases(v['operating_area'])
        if not phrases:
            rows.append({'vessel_id': v['id'], 'vessel': v['name'],
                        'operating_area': v['operating_area'], 'phrase': None,
                        'matched_name': None, 'place_type': None, 'mrgid': None, 'bbox': None})
        for p in phrases:
            res = lookup_phrase(p)
            rows.append({'vessel_id': v['id'], 'vessel': v['name'],
                        'operating_area': v['operating_area'], 'phrase': p,
                        'matched_name': res['matched_name'] if res else None,
                        'place_type':   res['place_type'] if res else None,
                        'mrgid':        res['mrgid'] if res else None,
                        'bbox':         res['bbox'] if res else None})
        if (i + 1) % 25 == 0:
            print(f'  ...{i+1}/{len(oa)} vessels, {len(_cache)} unique phrases cached')

    results_df = pd.DataFrame(rows)
    hit = results_df['matched_name'].notna().sum()
    tot = results_df['phrase'].notna().sum()
    print(f'\nphrase matches: {hit}/{tot} ({round(100*hit/max(tot,1))}%)')
    results_df.head(30)


loaded cached results: 384 rows from /Users/adamgent/greenwater/notebooks/gazetteer_results.pkl
(delete that file to force a fresh API run)


In [13]:
results_df.to_pickle(RESULTS_PATH)
print(f'saved {len(results_df)} rows to {RESULTS_PATH}')

saved 384 rows to /Users/adamgent/greenwater/notebooks/gazetteer_results.pkl


In [22]:
!pip install shapely
import re, requests
from shapely import wkt
from shapely.geometry import mapping
from shapely.ops import unary_union

_poly_cache = {}
def fetch_polygon(mrgid, tol=0.1):
    """Fetch the real polygon for an MRGID, simplified. Returns a GeoJSON geometry dict."""
    if mrgid in _poly_cache:
        return _poly_cache[mrgid]
    gj = None
    try:
        d = requests.get(f'https://marineregions.org/rest/getGazetteerGeometries.jsonld/{int(mrgid)}/',
                        headers={'Accept': 'application/json'}, timeout=40).json()
        geoms = d['mr:hasGeometry']
        if isinstance(geoms, dict):
            geoms = [geoms]
        parts = []
        for g in geoms:
            w = g['gsp:asWKT']; w = w['@value'] if isinstance(w, dict) else w
            parts.append(wkt.loads(re.sub(r'^\s*<[^>]+>\s*', '', w)))   # strip <CRS> prefix
        geom = unary_union(parts).simplify(tol, preserve_topology=True)
        gj = mapping(geom)
    except Exception as e:
        print('geom fail', mrgid, e)
    _poly_cache[mrgid] = gj
    return gj

  Using cached shapely-2.1.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.8 kB)
Using cached shapely-2.1.2-cp311-cp311-macosx_11_0_arm64.whl (1.6 MB)


In [23]:
from ipyleaflet import Map, GeoJSON, basemaps
import time

m = Map(center=(20, 0), zoom=2, basemap=basemaps.CartoDB.Positron, scroll_wheel_zoom=True)
regions = results_df[results_df['mrgid'].notna()].drop_duplicates('mrgid')
print(f'fetching {len(regions)} polygons (cached after first run)…')
for _, r in regions.iterrows():
    gj = fetch_polygon(r['mrgid'], tol=0.1)   # raise tol -> coarser/lighter, lower -> more detail
    if not gj:
        continue
    m.add_layer(GeoJSON(
        data={'type': 'Feature', 'geometry': gj,
            'properties': {'name': r['matched_name'], 'phrase': r['phrase']}},
        style={'color': '#2A7B6F', 'weight': 1, 'fillOpacity': 0.12},
        hover_style={'fillOpacity': 0.4},
        name=str(r['matched_name'])))
    time.sleep(0.15)
m


fetching 65 polygons (cached after first run)…


Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [ ]:
!pip install ipyleaflet
from ipyleaflet import Map, Rectangle, basemaps
from ipywidgets import HTML

m = Map(center=(20, 0), zoom=2, basemap=basemaps.CartoDB.Positron, scroll_wheel_zoom=True)

matched = results_df[results_df['bbox'].notna()]
seen = set()
for _, r in matched.iterrows():
    bb = r['bbox']
    if not bb or None in bb:
        continue
    if r['mrgid'] in seen:          # one rectangle per region to cut clutter
        continue
    
    seen.add(r['mrgid'])
    min_lon, min_lat, max_lon, max_lat = bb
    rect = Rectangle(
        bounds=((min_lat, min_lon), (max_lat, max_lon)),
        color='#2A7B6F', weight=1, fill_opacity=0.08,
    )
    try:
        rect.popup = HTML(f"{r['phrase']} → {r['matched_name']} ({r['place_type']})")
    except Exception:
        pass
    m.add_layer(rect)

print(f'{len(seen)} unique regions drawn')
m

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.2 MB/s eta 0:00:00
54 unique regions drawn


Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [15]:
# --- side-by-side: raw operating_area  vs  gazetteer output ---
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 250)

def summarize(g):
    parts = []
    for p, n, t in zip(g['phrase'], g['matched_name'], g['place_type']):
        if p is None:
            continue
        parts.append(f"{p} -> {n} [{t}]" if n else f"{p} -> (no match)")
    return ' | '.join(parts)

compare = (results_df
           .groupby(['vessel_id', 'vessel', 'operating_area'])
           .apply(summarize)
           .reset_index(name='gazetteer_output'))

compare[['operating_area', 'gazetteer_output']]


/var/folders/vb/c_7gcdvd2kg4s_jdwfky3xhm0000gn/T/ipykernel_69044/725921185.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize)


,operating_area,gazetteer_output
0,"Coast of California, Oregon and Washington, extended cruses to Mexico, Hawaii and Alaska",Coast of California -> (no match) | Oregon -> (no match) | Washington -> Washington Strait [Strait] | extended cruses to Mexico -> (no match) | Hawaii -> Hawaii [Marine Province] | Alaska -> Gulf of Alaska [IHO Sea Area]
1,"Gulf of California, Norht Pacific, Honolulu,Kao-hsiung, Papeete, Wellongton Auckland",Gulf of California -> Gulf of California [IHO Sea Area] | Norht Pacific -> (no match) | Honolulu -> (no match) | Kao-hsiung -> (no match) | Papeete -> (no match) | Wellongton Auckland -> (no match)
2,North Sea,North Sea -> North Sea [IHO Sea Area]
3,Great Lakes: Lake Superior,Great Lakes: Lake Superior -> (no match)
4,"North American coast from Nova Scotia to the Caribbean, and beyond Bermuda",North American coast from Nova Scotia to the Caribbean -> (no match) | beyond Bermuda -> (no match)
5,off San Diego,off San Diego -> (no match)
6,"Monterey & San Francisco Bays, coastal California, entire West Coast",Monterey & San Francisco Bays -> (no match) | coastal California -> (no match) | entire West Coast -> (no match)
7,Alaska to Attu Island at the extreme west end of the Aleutian chain to southeast Alaska and north past St. Matthew island in the Bering sea.,north past St -> (no match) | Matthew island in the Bering sea -> (no match)
8,"North Atlantic and Mediterranean Sea in the area 10-60 N,\n40W-35E and on a case by case basis depending on annual\noperation plan.",North Atlantic -> North Atlantic Ocean [IHO Sea Area] | Mediterranean Sea in the area 10-60 N -> (no match) | 40W-35E -> (no match) | on a case by case basis depending on annual\noperation plan -> (no match)
9,"South China Sea, South-West Pacific",South China Sea -> South China Sea [IHO Sea Area] | South-West Pacific -> (no match)
